<div style="
    background: linear-gradient(90deg, #e8f4f3, #dff0ee);
    border: 2px solid #2c7f7b;
    padding: 20px;
    border-radius: 8px;
    font-size: 22px;
    text-align: center;
    margin-top: 30px;
    margin-bottom: 30px;
">
<strong>⚙️ Stage 1 — Setup and Run</strong><br><br>
Configure and launch the Carbonara pipeline.  

Prepare inputs, define the search, start the run, and initialise monitoring of backmapping and FOXS evaluation.
</div>


<div style="
    background-color: #eef6f6;
    border-left: 6px solid #2c7f7b;
    padding: 12px;
    border-radius: 6px;
    font-size: 16px;
">
<strong>🧠 Configure search:</strong><br><br>
This cell sets up the main Carbonara search algorithm.  

Provide a project name (<code>run_name</code>), along with the input structure (<code>pdb_name</code>) and SAXS data file (<code>saxs_name</code>).  
Optionally, include a PAE file (<code>pae_name</code>) to guide flexibility.  

The setup will then be initialised. See the README for the full range of configuration options.
</div>

In [ ]:
import CarbonaraDataTools as CDT
import numpy as np
import re
import fittingAnalysis as fa

# @title Run this to set the carbonara model up with a given run name
run_name = "humanSmarcal"
pdb_name = "pdbFiles/foldSmarcal.cif"
saxs_name = "saxsFiles/smrclcnc_a2.dat"
pae_name = "paeFiles/foldSmarcal.json"


import sys
!{sys.executable} setup_carbonara_allAtom.py -p $pdb_name -s $saxs_name -n $run_name 

# for use with pae file (f for flexibility)

#!python setup_carbonara_allAtom.py -p $pdb_name -s $saxs_name -f $pae_name -n $run_name --alphaFoldFlex

#set up foxs script
from pathlib import Path
foxs_cmd = f'python3 {Path("external/pyFoXS/pyFoXS/foxs.py").resolve()}'


<div style="
    background-color: #eef6f6;
    border-left: 6px solid #2c7f7b;
    padding: 12px;
    border-radius: 6px;
    font-size: 16px;
    margin-top: 10px;
    margin-bottom: 10px;
">
<strong>📈 Initial scattering check:</strong><br><br>
This cell runs an initial FoXS check using the starting structure and experimental SAXS data.  
It reports the initial <code>&chi;<sup>2</sup></code> value and shows the corresponding fit and residuals.<br><br>

This gives a quick indication of how well the starting model already agrees with the data.  
In general, a poorer initial fit may mean that Carbonara will need longer to find good-fitting conformations.  
The plot also helps identify where the main discrepancies lie across the scattering curve.<br><br>

You may optionally set a maximum fitting range <code>max_q</code>.  
This can be useful if you want to focus on larger-scale structural features carried by the low-<code>q</code> region.  
For example, the initial fit may improve substantially as the fitted range is restricted, which can indicate that the main mismatch lies at higher <code>q</code>.  
Although Carbonara typically uses <code>q = 0.2</code> as its default upper limit, you may prefer a smaller value such as <code>q = 0.1</code> if your main interest is in low-<code>q</code> behaviour.

In [ ]:
max_q=0.2

foxs_result = CDT.run_initial_foxs_check(
    pdb_name=pdb_name,
    saxs_name=saxs_name,
    foxs_cmd=foxs_cmd,
     max_q =max_q
)

In [ ]:
# if you want to fit on max_q <0.2 (say max_q =0.1) you set it here

CDT.toggle_startk("carbonara/RunMe_"+run_name+".sh",max_q)

<div style="
    background-color: #eef6f6;
    border-left: 6px solid #2c7f7b;
    padding: 12px;
    border-radius: 6px;
    font-size: 16px;
">
<strong>⚙️ Initialise run:</strong><br><br>
This cell starts the Carbonara pipeline and initialises the live monitoring system.  
As the search progresses, the monitor performs all-atom backmapping and FoXS evaluation on generated structures.<br><br>

Choose a <code>&chi;<sup>2</sup></code> threshold to define what constitutes a <em>high-quality</em> fit (for example, <code>threshold = 2.5</code> selects structures with <code>&chi;<sup>2</sup> &lt; 2.5</code>).<br><br>

The parameter <code>defer_backmap_seconds</code> controls how long the algorithm waits before beginning all-atom reconstruction.  
If the initial structure is a poor fit, early iterations often produce many intermediate improvements that are not yet high quality.  
Delaying backmapping (e.g. 600 seconds) reduces backlog and focuses computation on more promising structures.<br><br>

The parameter <code>every_s</code> sets how frequently the monitor updates (in seconds), controlling how often new results are checked and processed.
</div>

In [ ]:
from pathlib import Path
from run_frontend import CarbonaraRunner

runner = CarbonaraRunner(run_name, foxs_cmd=foxs_cmd)
runner.start()
runner.start_monitor(threshold=2.5, every_s=10, defer_backmap_seconds=600)

<div style="
    background-color: #fdecea;
    border-left: 6px solid #d9534f;
    padding: 12px;
    border-radius: 6px;
    font-size: 16px;
">
<strong>🛑 Terminate run:</strong><br><br>
Run this cell to stop Carbonara.  

It will be terminated.
</div>

In [ ]:
runner.stop()

<div style="
    background: linear-gradient(90deg, #f4f1fb, #e9e4f7);
    border: 2px solid #7a5cc;
    padding: 20px;
    border-radius: 8px;
    font-size: 22px;
    text-align: center;
    margin-top: 30px;
    margin-bottom: 30px;
">
<strong>📊 STAGE 2 — ANALYSIS</strong><br><br>
Analyse the generated predictions, quantify structural variation, and interpret ensemble behaviour.<br><br>

<div style="
    background-color: #ffffff;
    border-left: 5px solid #7a5cc;
    padding: 10px;
    font-size: 16px;
    border-radius: 5px;
">
<strong>⏱️ You can start this immediately.</strong><br>
There is no need to wait for the Carbonara run to finish — analysis can be performed while the algorithm is still running.
</div>
</div>

<div style="
    background-color: #f4f1fb;
    border-left: 6px solid #7a5cc2;
    padding: 14px 18px;
    border-radius: 6px;
    font-size: 16px;
    margin-top: 10px;
    margin-bottom: 10px;
">
<strong>📥 Collect predictions:</strong><br><br>
At this stage you can gather predictions in one of two ways.<br><br>

Use the <em>all predictions</em> option to collect every structure that passes the chosen <code>&chi;<sup>2</sup></code> threshold.  
This is useful for more detailed or advanced analysis, but may include many closely related structures, especially from the same run.<br><br>

Use the <em>best per run</em> option to collect just one representative structure from each run, chosen as the prediction whose <code>&chi;<sup>2</sup></code> is closest to 1.  
This gives a smaller, more manageable set and often provides a clearer overview of the distinct outcomes across runs.
</div>

In [ ]:
#Use all predictions option, just run

directory = "carbonara_runs/"+run_name
run = "fitdata"

good_preds = fa.collect_good_prediction_files(
    fitdata_dir=directory+"/"+run,
    chi2_threshold=2.5
)

print(f"Found {len(good_preds)} good predictions")
#for pdb, chi2 in good_preds[:len(good_preds)]:
 #   print(f"{chi2:.3f}  {pdb}")
highQualityPredictions= [pdb for pdb, chi2 in good_preds]

In [ ]:
#Best per run option, just run

directory = "carbonara_runs/"+run_name
run = "fitdata"

best_preds = fa.collect_best_prediction_per_run_closest_to_one(
    fitdata_dir=directory + "/" + run
)

for run_no, entry in enumerate(best_preds, start=1):
    if entry is None:
        print(f"Run {run_no}: no valid prediction yet")
    else:
        _, chi2 = entry
        print(f"Run {run_no}: chi² = {chi2:.3f}")

highQualityPredictions = [
    entry[0] if entry is not None else None
    for entry in best_preds
]

In [ ]:
highQualityPredictions

<div style="
    background-color: #f4f1fb;
    border-left: 6px solid #7a5cc2;
    padding: 14px 18px;
    border-radius: 6px;
    font-size: 16px;
    margin-top: 10px;
    margin-bottom: 10px;
">
<strong>📏 Compare structural variation:</strong><br><br>
Run these cells to quantify structural variation using RMSD or TM-score.  

This compares:
<ul style="margin-top:6px;">
<li>variation between predictions (ensemble diversity),</li>
<li>deviation from the original structure (extent of structural change).</li>
</ul>
</div>

In [ ]:
# Run to calculate the variation between predictions

validPreds = [p for p in highQualityPredictions if p is not None]

rmsdPairedComparisons = fa.pairwise_structure_metrics(
    validPreds,
    fa.compare_structures_vals
)

In [ ]:
# Run to calculate the variation between predictions and the original structure

originalPdb = "pdbFiles/foldSmarcal.cif"

validPreds = [p for p in highQualityPredictions if p is not None]

rmsdToOriginal = fa.structure_metrics_vs_carbonara(
    pdb_files=validPreds,
    carbonara_dir=directory,
    compare_func=fa.compare_structures_vals_carbonara
)

In [ ]:
# Run to view histograms of these values

metric = "rmsd"
#could be 'tm' or 'gdt_ts'


fa.plot_overlaid_histograms([rmsdPairedComparisons,rmsdToOriginal],
    "rmsd",
    ["Carbonara-Carbonara","Carbonara-Original"],
    "RMSD (Angstroms)",
    "Comparative variations in RMSD Score of IL predictions",
    bins=40,
    figsize=(4.5, 3.5),
    colors=["blue","red"],
    alpha=0.5,
    density=False,
    dpi=300
)

<div style="
    background-color: #f4f1fb;
    border-left: 6px solid #7a5cc2;
    padding: 14px 18px;
    border-radius: 6px;
    font-size: 16px;
    margin-top: 10px;
    margin-bottom: 10px;
">
<strong>📐 Analyse radius of gyration (Rg):</strong><br><br>
Run this cell to examine the distribution of Rg across predictions.  

This includes both individual structures and mixtures of predictions (ensembles), allowing you to assess overall compactness and structural spread.
</div>

In [ ]:
# you can look in the variation in Radius of Gyration also
validPreds = [p for p in highQualityPredictions if p is not None]

rg_results = fa.calc_rg_distribution(
    pdb_files=validPreds,
    rg_func=fa.radius_of_gyration,
    weighted=False
)

# Rg of the original PDB
originalPdb = "pdbFiles/foldSmarcal.cif"

rg_results_og = fa.calc_rg_distribution(
    pdb_files=[originalPdb],
    rg_func=fa.radius_of_gyration,
    weighted=False
)

In [ ]:
# to view the Rg histograms

metric = "rg"

fa.plot_overlaid_histograms([rg_results,rg_results_og],
    metric,
    ["carbonra predictions","original model"],
    "Rg (Angstroms)",
    "Variation in Rg of carbonara predictions compared to homology model",
    bins=40,
    figsize=(4.5, 3.5),
    colors=["blue","red"],
    alpha=0.5,
    density=False,
    dpi=300
)

<div style="
    background-color: #f4f1fb;
    border-left: 6px solid #7a5cc2;
    padding: 14px 18px;
    border-radius: 6px;
    font-size: 16px;
    margin-top: 10px;
    margin-bottom: 10px;
">
<strong>🧬 Visualise structures (3D):</strong><br><br>
Run this cell to explore predictions using 3D cartoon representations (requires that pymol3d installed)

You can view individual structures, compare predictions, and overlay them with the original structure to assess structural differences.
</div>

In [ ]:
#View a prediction, check it exists first ! 


fa.visualisePredictionIndividual(highQualityPredictions[2])

In [ ]:
# compare the prediction to the original (which is shown slighly opaque)

fa.visualisePredictionComp(highQualityPredictions[2],pdb_name, do_superpose=True)

In [ ]:
#compare side by side with scattering fit

fa.show_structure_and_foxs_side_by_side(
    pdb_name=highQualityPredictions[2],
    saxs_name=saxs_name,
    foxs_cmd=foxs_cmd,
    max_q=0.2,
)

In [ ]:
# to see a set of stcutures, warning this could take time if there are a lot, you can subset e.g. highQualityPredictions[1:10]

validPreds = [p for p in highQualityPredictions if p is not None]

fa.visualisePredictionComp_panel(
    validPreds,
    pdb_name,
    do_superpose=True,
    ncols=3
)